# 🧱 Backfill (What it is and why it matters)

**Context:**
This step **comes before** your Phase 4 analysis notebook. Phase 4 expects all champion time-series, split boundaries, and full history to be available in a **Series Store (Parquet “lake”)**.
**Phase 3 (Backfill)** builds that store from raw experiment outputs (your per-run JSON folders).

---

## 👀 What this step does (in your pipeline)

1. **Discovers experiment runs** in your campaign group(s) (e.g., `HPO_210_Lag_100_Horizon_300`), optionally limited by **families** (e.g., `seq2`, `arps`).
2. **Reads raw JSON results** from each run (per job / per well).
3. **Normalizes & validates series** (time `t`, predictions `yhat`, optional `ytrue`) and **writes Parquet** per job:

   ```
   series_store/
     group=G/arch=A[/dataset=D][/well=W]/campaign=C/job=HASH.parquet
   ```
4. **Upserts metadata**:

   * `meta/boundaries.parquet` → **train/val/test** split indices per well
   * `history/well=<well>/history.parquet` → full ground truth **(t, ytrue)**
5. **Reports a compact summary** of actions (CREATED / UPDATED / SKIPPED / FAILED) and checks on-disk truth for `boundaries.parquet` and `history/*`.

> After this notebook, your **Phase 4** notebook can load everything fast, reproducibly, and without touching raw JSON again.

---

## 🔗 How it connects to Phase 4

* **Phase 4 needs:**

  * Champion **series** per job → to build **intra** (per-family) and **inter** (final) ensembles
  * **Boundaries** → to align Train/Val/Test regions
  * **Full history** → to map index → time (`global_idx → t`), compute metrics, and draw plots

* **Phase 3 produces exactly those files**, so Phase 4 can do:

  * Conjugated ensemble plots
  * Members “spaghetti” plots
  * Post-ensemble metrics and risk CDFs

---

## 🧩 What you configure (just a few knobs)

* **Campaign groups** to scan: `CAMPAIGN_GROUP_NAMES = ["HPO_210_Lag_100_Horizon_300"]`
* **Optional family filter**: `FAMILIES = {"seq2", "arps"}  # or None`
* **Series Store root**: `SERIES_STORE_ROOT = Path("series_store")`
* **Safety**: `DRY_RUN = True` to preview write actions
* **Speed**: `CONCURRENCY = 4` (IO-bound; can increase on fast disks)

---

## 📦 What it reads vs. what it writes

**Reads**

* Experiment results at:
  `src/experiment_configs/<CAMPAIGN_GROUP>/results/<family>/<run>/results/**/*.json`

**Writes (idempotent)**

* Per-job series Parquet:
  `series_store/group=.../arch=.../dataset=.../well=.../campaign=.../job=<hash>.parquet`
* **Boundaries manifest** (one file):
  `series_store/meta/boundaries.parquet`
* **Per-well history**:
  `series_store/history/well=<well>/history.parquet`

If files already exist, it **skips** or **updates** safely (the summary will tell you).

---

## ✅ What you’ll see when it works

* A short **run list** (first 8 paths)
* One **action row per job** (CREATED / UPDATED / SKIPPED / FAILED)
* A small **preview table**
* **On-disk checks**:

  * `boundaries.parquet` rows/columns
  * `history/` well folders
* **Quick JSON diagnostics** for a sample run (do your JSONs include boundaries/full_history?)

---

## 🧠 When to run it

* New experiments finished running
* You changed the JSON schema / extraction logic
* You want to rebuild or validate the store
* You’re preparing to run **Phase 4**

> Tip: start with `DRY_RUN=True` to confirm actions, then set it to `False`.

---

## 🚑 Common gotchas (and what the notebook already handles)

* **Missing campaign path** → warned, others continue
* **Old `run_backfill` signatures** → wrapper calls the right one
* **Inconsistent metadata** → logged; boundaries/history are upserted idempotently
* **Large runs** → increase `CONCURRENCY`; the work is IO-bound

---

## 🏁 Bottom line

Run **Phase 3 Backfill** first to **materialize** a clean, queryable **Series Store**.
Then run **Phase 4** to select champions, build ensembles, compute metrics, and visualize—all on top of the same, versionable data lake.


In [ ]:
from __future__ import annotations
from forecast_pipeline.io_utils import configure_logging
configure_logging()
# =========================
# USER CONFIG — edit me
# =========================
from pathlib import Path

CAMPAIGN_GROUP_NAMES = [
    "HPO_153_Lag_100_Horizon_150",
]

# Families filter (None = all). Example: {"arps", "seq2", "darts"}
FAMILIES: set[str] | None = None

# Where to write Series Store (relative to project root)
SERIES_STORE_ROOT = Path("series_store")

# Safety: DRY_RUN=True will not write; just report
DRY_RUN = False

# Parallel workers
CONCURRENCY = 8


# =========================
# Minimal logging (terse)
# =========================
import sys, logging
from collections import Counter

def _configure_minimal_logging(level=logging.INFO) -> None:
    root = logging.getLogger()
    for h in root.handlers[:]:
        root.removeHandler(h)
    logging.basicConfig(level=level, format="%(levelname)s - %(message)s")

_configure_minimal_logging()
log = logging.getLogger("phase3_backfill")


# =========================
# Path resolution
# =========================
def _find_project_root(start_path: Path, marker: str = "src") -> Path:
    cur = start_path.resolve()
    while cur != cur.parent:
        if (cur / marker).is_dir():
            return cur
        cur = cur.parent
    raise FileNotFoundError(f"Could not find project root with marker '{marker}' from {start_path}")

try:
    PROJECT_ROOT = _find_project_root(Path.cwd(), marker="src")
except FileNotFoundError as e:
    print(f"ERROR: {e}\nFalling back to parent of CWD (may be wrong).")
    PROJECT_ROOT = Path.cwd().parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

EXPERIMENT_CONFIG_ROOT = SRC / "experiment_configs"
SERIES_STORE_ROOT_ABS = PROJECT_ROOT / SERIES_STORE_ROOT

print("— Paths —")
print(f"PROJECT_ROOT:           {PROJECT_ROOT}")
print(f"EXPERIMENT_CONFIG_ROOT: {EXPERIMENT_CONFIG_ROOT}")
print(f"SERIES_STORE_ROOT:      {SERIES_STORE_ROOT_ABS}")
print(f"DRY_RUN:                {DRY_RUN}")
print(f"CONCURRENCY:            {CONCURRENCY}")


# =========================
# Imports from your codebase
# =========================
from common.series_backfill import run_backfill, quick_meta_diagnostics  # <- your refactored helpers


# =========================
# Discover run directories
# =========================
def _discover_run_dirs(groups: list[str], families: set[str] | None) -> list[Path]:
    out: list[Path] = []
    for g in groups:
        base = EXPERIMENT_CONFIG_ROOT / g / "results"
        if not base.is_dir():
            log.warning("Group results not found: %s", base)
            continue

        for fam_dir in sorted(p for p in base.iterdir() if p.is_dir()):
            fam = fam_dir.name
            if families and fam not in families:
                continue
            for run_dir in fam_dir.iterdir():
                if run_dir.is_dir() and (run_dir / "results").is_dir():
                    out.append(run_dir)
    return sorted(out)

all_run_base_dirs = _discover_run_dirs(CAMPAIGN_GROUP_NAMES, FAMILIES)
print(f"Discovered {len(all_run_base_dirs)} run(s).")
for p in all_run_base_dirs[:8]:
    print("  •", p)
if len(all_run_base_dirs) > 8:
    print(f"  … (+{len(all_run_base_dirs)-8} more)")


# =========================
# Execute backfill (robust to signature changes)
# =========================
def _call_run_backfill(
    *,
    results_root: Path,
    series_store_root: Path,
    concurrency: int,
    dry_run: bool,
):
    """
    Some versions of run_backfill accept a `logger` kwarg, others do not.
    This wrapper detects support and calls accordingly.
    """
    try:
        # Try with logger first (newer refactor)
        return run_backfill(
            results_root=results_root,
            series_store_root=series_store_root,
            concurrency=concurrency,
            dry_run=dry_run,
        )
    except TypeError:
        # Fallback to older signature without logger
        return run_backfill(
            results_root=results_root,
            series_store_root=series_store_root,
            concurrency=concurrency,
            dry_run=dry_run,
        )

all_results = []
if not all_run_base_dirs:
    print("Nothing to process.")
else:
    for run_base_dir in all_run_base_dirs:
        print("\n" + "═"*70)
        print(f"Processing: {run_base_dir.name}")
        print("═"*70)
        res = _call_run_backfill(
            results_root=run_base_dir,
            series_store_root=SERIES_STORE_ROOT_ABS,
            concurrency=CONCURRENCY,
            dry_run=DRY_RUN,
        )
        all_results.extend(res)

print("\nBackfill complete.")


# =========================
# Summary (concise + table)
# =========================
import pandas as pd

counts = Counter(r.action for r in all_results)
total = len(all_results)

print("\n— Global Summary —")
print(f"Total actions:                {total}")
print(f"  Created:                    {counts.get('CREATED', 0)}")
print(f"  Updated:                    {counts.get('UPDATED', 0)}")
print(f"  Skipped:                    {counts.get('SKIPPED', 0)}")
print(f"  Failed:                     {counts.get('FAILED', 0)}")
# handle either DRY_RUN or DRY_RUN_CREATE labels gracefully
print(f"  Would Create (Dry Run):     {counts.get('DRY_RUN_CREATE', 0) + counts.get('DRY_RUN', 0)}")

df = pd.DataFrame([{
    "job_hash": r.job_hash,
    "action": r.action,
    "reason": r.reason,
    "path": str(r.parquet_path),
} for r in all_results])

if not df.empty:
    try:
        display(df.head(20).style.hide(axis="index"))
    except Exception:
        # older pandas without .hide
        display(df.head(20))
else:
    print("No actions recorded.")


# =========================
# Filesystem truth check (no illusions)
# =========================
bnd = SERIES_STORE_ROOT_ABS / "meta" / "boundaries.parquet"
bnd_ok = bnd.exists()

print("\n— On-disk Meta —")
print(f"boundaries.parquet:       {'OK' if bnd_ok else 'MISSING'}")
if bnd_ok:
    try:
        bdf = pd.read_parquet(bnd)
        print(f"  rows={len(bdf)}  cols={list(bdf.columns)}")
    except Exception as e:
        print("  (could not read boundaries.parquet):", e)

hist_root = SERIES_STORE_ROOT_ABS / "history"
if hist_root.is_dir():
    wells = sorted(p.name for p in hist_root.iterdir() if p.is_dir())
    preview = wells[:8]
    print("history wells (preview):", ", ".join(preview) or "(none)")
else:
    print("history/: MISSING")


# =========================
# Quick JSON diagnostics
# =========================
if all_run_base_dirs:
    sample_run = all_run_base_dirs[0]
    print("\nQuick meta diagnostics on:", sample_run)
    # This prints counts and on-disk existence in a compact way
    quick_meta_diagnostics(sample_run, SERIES_STORE_ROOT_ABS)

print("\nDone.")
